# **<span style="color:#48B3AF; font-family: 'Courier New'"> Report </span>**

## **<span style="color:#476EAE; font-family: 'Courier New"> Resources </span>**

In [2]:
# Infrastructure
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
drug_coverage = pd.read_csv("drug_coverage.csv")
df_mapping = pd.read_csv("fuzzymatching.csv")

In [5]:
drug_coverage.head(2)

,Unnamed: 0,source,atc_code,dci_name,dc_name,is_covered,manufacturer_name,manufacturer_label,country
0,502,md,A02BA03,FAMOTIDINUM,Quamatel® Mini,True,Gedeon Richter SA,Gedeon Richter,Ungaria
1,1212,md,A02BA03,FAMOTIDINUM,Famotidin-BP,True,"SC Balkan Pharmaceuticals SRL, Republica Moldo...",Balkan Pharmaceuticals,Republica Moldova


In [13]:
drug_coverage[drug_coverage["source"] == 'md'].value_counts(["manufacturer_label"])

manufacturer_label                
Balkan Pharmaceuticals                169
KRKA                                   82
Gedeon Richter                         56
Nobel Ilac Sanayii ve Ticaret A.Ş.     49
Zentiva                                37
                                     ... 
Liqvor                                  1
Terapia                                 1
SmithKline Beecham                      1
Unique Pharmaceutical Laboratories      1
Zim Laboratories                        1
Name: count, Length: 134, dtype: int64

In [19]:
import textwrap
import pandas as pd
import plotly.express as px

# Count manufacturers only for md
counts = (
    drug_coverage.loc[drug_coverage["source"] == "md", "manufacturer_label"]
    .value_counts()
    .loc[lambda s: s > 15]
    .sort_values(ascending=True)
    .reset_index()
)

counts.columns = ["manufacturer_label", "n_drugs"]

# Wrap labels into max 2 lines
def wrap_two_lines(text, width=22):
    lines = textwrap.wrap(text, width=width)
    if len(lines) <= 2:
        return "<br>".join(lines)
    return "<br>".join([lines[0], " ".join(lines[1:])])

counts["manufacturer_label_wrapped"] = counts["manufacturer_label"].apply(wrap_two_lines)

fig = px.bar(
    counts,
    x="n_drugs",
    y="manufacturer_label",
    orientation="h",
    text="n_drugs",
    color="n_drugs",
    color_continuous_scale="Tealgrn",
    title="Manufacturers with More Than 15 Covered Drugs (MD)"
)

fig.update_traces(
    textposition="outside",
    cliponaxis=False
)

fig.update_layout(
    template="plotly_white",
    height=max(500, len(counts) * 32),
    showlegend=False,
    coloraxis_showscale=False,
    xaxis_title="Number of drugs",
    yaxis_title="Manufacturer",
    margin=dict(l=180, r=30, t=60, b=40)
)

fig.update_yaxes(
    categoryorder="array",
    categoryarray=counts["manufacturer_label"],
    tickvals=counts["manufacturer_label"],
    ticktext=counts["manufacturer_label_wrapped"]
)

fig.update_xaxes(
    range=[0, counts["n_drugs"].max() * 1.12]
)

fig.show()

In [29]:
import textwrap
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def wrap_two_lines(text, width=22):
    lines = textwrap.wrap(str(text), width=width)
    if len(lines) <= 2:
        return "<br>".join(lines)
    return "<br>".join([lines[0], " ".join(lines[1:])])

def prepare_counts(df, source, min_count, top_n=10):
    counts = (
        df.loc[df["source"] == source, "manufacturer_label"]
        .value_counts()
        .loc[lambda s: s > min_count]
        .reset_index()
    )
    counts.columns = ["manufacturer_label", "n_drugs"]

    counts = (
        counts
        .sort_values("n_drugs", ascending=False)
        .head(top_n)
        .sort_values("n_drugs", ascending=True)
        .reset_index(drop=True)
    )

    counts["manufacturer_label_wrapped"] = counts["manufacturer_label"].apply(wrap_two_lines)
    return counts

# Prepare top 10 for each country
counts_md = prepare_counts(drug_coverage, source="md", min_count=15, top_n=10)
counts_es = prepare_counts(drug_coverage, source="es", min_count=500, top_n=10)

# Create subplot
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "TOP10 drug manufacturers - MD",
        "TOP10 drug manufacturers - ES"
    ),
    horizontal_spacing=0.18
)

# MD trace
fig.add_trace(
    go.Bar(
        x=counts_md["n_drugs"],
        y=counts_md["manufacturer_label"],
        orientation="h",
        text=counts_md["n_drugs"],
        textposition="outside",
        cliponaxis=False,
        marker=dict(
            color=counts_md["n_drugs"],
            colorscale="Tealgrn",
            line=dict(color="rgba(0,0,0,0.25)", width=1)
        ),
        hovertemplate="<b>%{y}</b><br>Drugs: %{x}<extra></extra>",
        showlegend=False
    ),
    row=1, col=1
)

# ES trace
fig.add_trace(
    go.Bar(
        x=counts_es["n_drugs"],
        y=counts_es["manufacturer_label"],
        orientation="h",
        text=counts_es["n_drugs"],
        textposition="outside",
        cliponaxis=False,
        marker=dict(
            color=counts_es["n_drugs"],
            colorscale="Tealgrn",
            line=dict(color="rgba(0,0,0,0.25)", width=1)
        ),
        hovertemplate="<b>%{y}</b><br>Drugs: %{x}<extra></extra>",
        showlegend=False
    ),
    row=1, col=2
)

# Wrapped y-axis labels
fig.update_yaxes(
    tickvals=counts_md["manufacturer_label"],
    ticktext=counts_md["manufacturer_label_wrapped"],
    categoryorder="array",
    categoryarray=counts_md["manufacturer_label"],
    title_text="Manufacturer",
    row=1, col=1
)

fig.update_yaxes(
    tickvals=counts_es["manufacturer_label"],
    ticktext=counts_es["manufacturer_label_wrapped"],
    categoryorder="array",
    categoryarray=counts_es["manufacturer_label"],
    title_text="Manufacturer",
    row=1, col=2
)

# X-axis ranges with room for outside labels
fig.update_xaxes(
    title_text="Number of drugs",
    range=[0, counts_md["n_drugs"].max() * 1.15],
    row=1, col=1
)

fig.update_xaxes(
    title_text="Number of drugs",
    range=[0, counts_es["n_drugs"].max() * 1.15],
    row=1, col=2
)

# Layout
fig.update_layout(
    template="plotly_white",
    title="Top manufacturers by covered drugs: MD vs ES",
    height=max(650, max(len(counts_md), len(counts_es)) * 42),
    margin=dict(l=160, r=40, t=90, b=50),
    showlegend=False
)

fig.show()

In [ ]:
import textwrap
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def wrap_two_lines(text, width=22):
    lines = textwrap.wrap(str(text), width=width)
    if len(lines) <= 2:
        return "<br>".join(lines)
    return "<br>".join([lines[0], " ".join(lines[1:])])

def prepare_counts(df, source, min_count, top_n=10):
    counts = (
        df.loc[df["source"] == source, "country"]
        .value_counts()
        .loc[lambda s: s > min_count]
        .reset_index()
    )
    counts.columns = ["country", "n_drugs"]

    counts = (
        counts
        .sort_values("n_drugs", ascending=False)
        .head(top_n)
        .sort_values("n_drugs", ascending=True)
        .reset_index(drop=True)
    )

    counts["country_wrapped"] = counts["country"].apply(wrap_two_lines)
    return counts

# Prepare top 10 for each country
counts_md = prepare_counts(drug_coverage, source="md", min_count=15, top_n=10)
counts_es = prepare_counts(drug_coverage, source="es", min_count=500, top_n=10)

# Create subplot
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "TOP10 countries of \ndrug manufacturers - MD",
        "TOP10 countries of \ndrug manufacturers - ES"
    ),
    horizontal_spacing=0.18
)

# MD trace
fig.add_trace(
    go.Bar(
        x=counts_md["n_drugs"],
        y=counts_md["country"],
        orientation="h",
        text=counts_md["n_drugs"],
        textposition="outside",
        cliponaxis=False,
        marker=dict(
            color=counts_md["n_drugs"],
            colorscale="Tealgrn",
            line=dict(color="rgba(0,0,0,0.25)", width=1)
        ),
        hovertemplate="<b>%{y}</b><br>Drugs: %{x}<extra></extra>",
        showlegend=False
    ),
    row=1, col=1
)

# ES trace
fig.add_trace(
    go.Bar(
        x=counts_es["n_drugs"],
        y=counts_es["country"],
        orientation="h",
        text=counts_es["n_drugs"],
        textposition="outside",
        cliponaxis=False,
        marker=dict(
            color=counts_es["n_drugs"],
            colorscale="Tealgrn",
            line=dict(color="rgba(0,0,0,0.25)", width=1)
        ),
        hovertemplate="<b>%{y}</b><br>Drugs: %{x}<extra></extra>",
        showlegend=False
    ),
    row=1, col=2
)

# Wrapped y-axis labels
fig.update_yaxes(
    tickvals=counts_md["country"],
    ticktext=counts_md["country_wrapped"],
    categoryorder="array",
    categoryarray=counts_md["country"],
    title_text="Country",
    row=1, col=1
)

fig.update_yaxes(
    tickvals=counts_es["country"],
    ticktext=counts_es["country_wrapped"],
    categoryorder="array",
    categoryarray=counts_es["country"],
    title_text="Country",
    row=1, col=2
)

# X-axis ranges with room for outside labels
fig.update_xaxes(
    title_text="Number of drugs",
    range=[0, counts_md["n_drugs"].max() * 1.15],
    row=1, col=1
)

fig.update_xaxes(
    title_text="Number of drugs",
    range=[0, counts_es["n_drugs"].max() * 1.15],
    row=1, col=2
)

# Layout
fig.update_layout(
    template="plotly_white",
    title="Top manufacturers' countries by covered drugs: MD vs ES",
    height=max(650, max(len(counts_md), len(counts_es)) * 42),
    margin=dict(l=160, r=40, t=90, b=50),
    showlegend=False
)

fig.show()

In [32]:
import plotly.graph_objects as go

fig = go.Figure()

# Square 1 at (1,1) to (2,2)
fig.add_trace(go.Scatter(
    x=[1, 2, 2, 1, 1],
    y=[1, 1, 2, 2, 1],
    mode='lines',
    fill='toself',
    fillcolor='rgba(149, 219, 229, 1)',
    line=dict(color='rgba(149, 219, 229, 1)'),
    hoverinfo='skip',
    showlegend=False
))

# Square 2 at (3,1) to (4,2)
fig.add_trace(go.Scatter(
    x=[3, 4, 4, 3, 3],
    y=[1, 1, 2, 2, 1],
    mode='lines',
    fill='toself',
    fillcolor='rgba(7, 130, 130, 1.0)',
    line=dict(color='rgba(7, 130, 130, 1.0)'),
    hoverinfo='skip',
    showlegend=False
))


# Add text annotations inside each square (centered)
fig.add_annotation(
    x=1.5, y=1.5,  # Center of first square
    text="1396 cases",
    showarrow=False,
    font=dict(color="white", size=20)
)
fig.add_annotation(
    x=1.5, y=0.75,  # Center of first square
    text="atc_exact_best_score",
    showarrow=False,
    font=dict(color="black", size=20)
)

fig.add_annotation(
    x=3.5, y=1.5,  # Center of second square
    text="4,7% fails",
    showarrow=False,
    font=dict(color="white", size=20)
)
fig.add_annotation(
    x=3.5, y=0.75,  # Center of second square
    text="no_candidates",
    showarrow=False,
    font=dict(color="black", size=20)
)

fig.add_annotation(
    x=5.5, y=1.5,  # Center of third square
    text="2 cases",
    showarrow=False,
    font=dict(color="white", size=20)
)

# Set layout to hide axes and grid for a clean look
fig.update_layout(
    xaxis=dict(visible=False, range=[0, 7]),
    yaxis=dict(visible=False, range=[0, 3]),
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(l=0, r=0, t=0, b=0)
)

fig.show()   

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# --------------------------------------------------
# 1. Prepare top 10 destination countries per source
# --------------------------------------------------
# Assumption:
# - drug_coverage has a country column with destination country names
# - replace "country" below with your real column name if different
#   e.g. "country_name", "country_label", "ma_country", etc.

country_col = "country"   # <-- change this if needed

top_md = (
    drug_coverage.loc[
        (drug_coverage["source"] == "md") &
        (drug_coverage[country_col].notna()),
        country_col
    ]
    .value_counts()
    .head(10)
    .reset_index()
)
top_md.columns = ["country", "n_md"]

top_es = (
    drug_coverage.loc[
        (drug_coverage["source"] == "es") &
        (drug_coverage[country_col].notna()),
        country_col
    ]
    .value_counts()
    .head(10)
    .reset_index()
)
top_es.columns = ["country", "n_es"]

# --------------------------------------------------
# 2. Country mapping: names -> ISO3 + lat/lon
#    Add/edit names if your data uses variants
# --------------------------------------------------
country_ref = {
    "Moldova": {"iso3": "MDA", "lat": 47.4116, "lon": 28.3699},
    "Spain": {"iso3": "ESP", "lat": 40.4637, "lon": -3.7492},

    "Romania": {"iso3": "ROU", "lat": 45.9432, "lon": 24.9668},
    "Germany": {"iso3": "DEU", "lat": 51.1657, "lon": 10.4515},
    "India": {"iso3": "IND", "lat": 20.5937, "lon": 78.9629},
    "Turkey": {"iso3": "TUR", "lat": 38.9637, "lon": 35.2433},
    "Hungary": {"iso3": "HUN", "lat": 47.1625, "lon": 19.5033},
    "Slovenia": {"iso3": "SVN", "lat": 46.1512, "lon": 14.9955},
    "Poland": {"iso3": "POL", "lat": 51.9194, "lon": 19.1451},
    "France": {"iso3": "FRA", "lat": 46.2276, "lon": 2.2137},
    "Italy": {"iso3": "ITA", "lat": 41.8719, "lon": 12.5674},
    "Netherlands": {"iso3": "NLD", "lat": 52.1326, "lon": 5.2913},
    "Belgium": {"iso3": "BEL", "lat": 50.5039, "lon": 4.4699},
    "United Kingdom": {"iso3": "GBR", "lat": 55.3781, "lon": -3.4360},
    "UK": {"iso3": "GBR", "lat": 55.3781, "lon": -3.4360},
    "Switzerland": {"iso3": "CHE", "lat": 46.8182, "lon": 8.2275},
    "Austria": {"iso3": "AUT", "lat": 47.5162, "lon": 14.5501},
    "Czech Republic": {"iso3": "CZE", "lat": 49.8175, "lon": 15.4730},
    "Czechia": {"iso3": "CZE", "lat": 49.8175, "lon": 15.4730},
    "Bulgaria": {"iso3": "BGR", "lat": 42.7339, "lon": 25.4858},
    "Portugal": {"iso3": "PRT", "lat": 39.3999, "lon": -8.2245},
    "Greece": {"iso3": "GRC", "lat": 39.0742, "lon": 21.8243},
    "Croatia": {"iso3": "HRV", "lat": 45.1000, "lon": 15.2000},
    "Serbia": {"iso3": "SRB", "lat": 44.0165, "lon": 21.0059},
    "Ukraine": {"iso3": "UKR", "lat": 48.3794, "lon": 31.1656},
    "China": {"iso3": "CHN", "lat": 35.8617, "lon": 104.1954},
    "United States": {"iso3": "USA", "lat": 37.0902, "lon": -95.7129},
    "USA": {"iso3": "USA", "lat": 37.0902, "lon": -95.7129}
}

# --------------------------------------------------
# 3. Keep only mapped countries
# --------------------------------------------------
top_md = top_md[top_md["country"].isin(country_ref)].copy()
top_es = top_es[top_es["country"].isin(country_ref)].copy()

top_md["iso3"] = top_md["country"].map(lambda x: country_ref[x]["iso3"])
top_es["iso3"] = top_es["country"].map(lambda x: country_ref[x]["iso3"])

# --------------------------------------------------
# 4. Build choropleth layer
#    category:
#    - Moldova top10 only
#    - Spain top10 only
#    - Shared in both top10
#    - Origins: Moldova, Spain
# --------------------------------------------------
md_set = set(top_md["country"])
es_set = set(top_es["country"])

all_countries = sorted(md_set | es_set | {"Moldova", "Spain"})

def classify_country(c):
    if c == "Moldova":
        return "Origin: Moldova"
    if c == "Spain":
        return "Origin: Spain"
    if c in md_set and c in es_set:
        return "Shared top10"
    if c in md_set:
        return "Moldova top10"
    if c in es_set:
        return "Spain top10"

map_df = pd.DataFrame({"country": all_countries})
map_df["category"] = map_df["country"].apply(classify_country)
map_df["iso3"] = map_df["country"].map(lambda x: country_ref[x]["iso3"])

color_map = {
    "Origin: Moldova": "#8E44AD",
    "Origin: Spain": "#E67E22",
    "Shared top10": "#16A085",
    "Moldova top10": "#5DADE2",
    "Spain top10": "#F5B041"
}

fig = px.choropleth(
    map_df,
    locations="iso3",
    locationmode="ISO-3",
    color="category",
    hover_name="country",
    color_discrete_map=color_map,
    projection="natural earth"
)

# --------------------------------------------------
# 5. Add connection lines
# --------------------------------------------------
moldova_lat = country_ref["Moldova"]["lat"]
moldova_lon = country_ref["Moldova"]["lon"]
spain_lat = country_ref["Spain"]["lat"]
spain_lon = country_ref["Spain"]["lon"]

for _, row in top_md.iterrows():
    fig.add_trace(
        go.Scattergeo(
            lon=[moldova_lon, country_ref[row["country"]]["lon"]],
            lat=[moldova_lat, country_ref[row["country"]]["lat"]],
            mode="lines+markers",
            line=dict(width=1.8, color="#5DADE2"),
            marker=dict(size=4, color="#5DADE2"),
            opacity=0.75,
            hovertemplate=f"Moldova → {row['country']}<br>Count: {row['n_md']}<extra></extra>",
            showlegend=False
        )
    )

for _, row in top_es.iterrows():
    fig.add_trace(
        go.Scattergeo(
            lon=[spain_lon, country_ref[row["country"]]["lon"]],
            lat=[spain_lat, country_ref[row["country"]]["lat"]],
            mode="lines+markers",
            line=dict(width=1.8, color="#F5B041"),
            marker=dict(size=4, color="#F5B041"),
            opacity=0.75,
            hovertemplate=f"Spain → {row['country']}<br>Count: {row['n_es']}<extra></extra>",
            showlegend=False
        )
    )

# --------------------------------------------------
# 6. Add emphasized origin markers
# --------------------------------------------------
fig.add_trace(
    go.Scattergeo(
        lon=[moldova_lon, spain_lon],
        lat=[moldova_lat, spain_lat],
        #text=["Moldova", "Spain"],
        mode="markers+text",
        textposition="top center",
        marker=dict(size=10, color=["#8E44AD", "#E67E22"], line=dict(color="black", width=0.8)),
        hovertemplate="%{text}<extra></extra>",
        showlegend=False
    )
)

# --------------------------------------------------
# 7. Layout
# --------------------------------------------------
fig.update_geos(
    showcountries=True,
    countrycolor="white",
    showcoastlines=True,
    coastlinecolor="lightgray",
    showland=True,
    landcolor="#F8F9F9",
    showocean=True,
    oceancolor="#EBF5FB",
    fitbounds="locations"
)

fig.update_layout(
    title="Connections from Moldova and Spain to Top 10 Countries",
    template="plotly_white",
    width=1400,
    height=800,
    margin=dict(l=10, r=10, t=60, b=80),
    legend_title_text="Country group",
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.10,
        xanchor="center",
        x=0.5,
        title_text=""
    )
)

# --------------------------------------------------
# 7. Add country names
# --------------------------------------------------
label_df = map_df.copy()
label_df["lat"] = label_df["country"].map(lambda x: country_ref[x]["lat"])
label_df["lon"] = label_df["country"].map(lambda x: country_ref[x]["lon"])

fig.add_trace(
    go.Scattergeo(
        lon=label_df["lon"],
        lat=label_df["lat"],
        text=label_df["country"],
        mode="text",
        textfont=dict(size=10, color="black"),
        hoverinfo="skip",
        showlegend=False
    )
)

fig.show()